## The differences in error
Here in this section we calculate the differences in errors from the base forecast errors, for all hierarchies.

In [ ]:
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root

import pandas as pd
import plotly.express as px
import python.utils as utils
import python.plots as plots
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS, TimeSeriesType
from python.aggregation import MONTHS, FORTNIGHTS, WEEKS, WEEKDAYS, WEEKENDS, DAYS
from python.temporal_hierarchy import TemporalHierarchy, MinimalTemporalHierarchy, SemanticTemporalHierarchy
from python.error_metrics import ErrorMetrics, MergedErrorMetrics, MetricsIndex, LoadedErrorMetrics, ErrorDifference, FilteredErrorMetrics, ClonedErrorMetrics, CustomMetrics
from python.error_metrics import MetricsIndex as MI # Alias
from python.forecast_reconciliation import ForecastReconciliation

In [ ]:
em = LoadedErrorMetrics('metrics/base_hier_errors.csv')
inc = pd.read_csv('metrics/base_hier_inconsistency.csv', index_col=0)

sth = SemanticTemporalHierarchy()
mth = MinimalTemporalHierarchy()
wth = TemporalHierarchy([MONTHS, WEEKS, DAYS], 'Minimal+Weeks')
wweth = TemporalHierarchy([MONTHS, WEEKS, WEEKDAYS, WEEKENDS, DAYS], 'Weekends+Weekdays')
wfth = TemporalHierarchy([MONTHS, FORTNIGHTS, WEEKS, DAYS], 'Weeks+Biweeks')

error_metric = 'nrmse'
em = FilteredErrorMetrics(em, MI(error_metric='nrmse'))

diffs = MergedErrorMetrics([
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=sth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=wth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=wweth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=mth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=wfth)),
])

nrmse_em, nrmse_diffs = em, diffs

### Checking the statistics of the diffs

In [ ]:
nrmse_diffs.get_error_stats(MI(granularity=DAYS))

On daily level the median of the change is slightly positive in all hierarchies, meaning more than half the time, the errors increased. Due to the outliers in the positive range the mean and variance are meaningless to look at. Note that it does make sense that there are no outliers with negative sign.

The smallest median difference is with `Minimal` hierarchy. Also this hierarchy does not have any outliers in terms of error difference, proven by the max and also the 90th percentile.

In [ ]:
nrmse_diffs.get_error_stats(MI(granularity=MONTHS))

On months level the median change is always negative, meaning more than half the cases, there was in improvement made to the errors.

The best median improvements on monthly granularity come from the `Semantic` hierarchy. It also has the smallest 90th percentile.

### Plotting the diffs
I will plot the differences of the errors the following way: on the X axis will be the base error values and on the y the difference made to the base error in a specific hierarchy, after filtering both axes for below the 99th quantile (to make the plots more understandable)

In [ ]:
for hier in (sth, mth, wth, wweth, wfth):
    plots.plot_error_diff(nrmse_em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth, granularity=DAYS),
                                    MI(forecast_type=RECONCILED_FORECASTS, hierarchy=hier, granularity=DAYS,),
                                    outlier_threshold=0.995
    ).show()

It's interesting that the negative differences are very low in absolute value and quite sparse compared to how it is in the monthly metrics (next section). It seems it's hard for the reconciliation to reconcile these high granularity samples. There almost aren't any negative differences which would match the x = -y axis (meaning the error was decreased to almost none)

When we look at the positive differences, all hierarchies look roughly the same. The negative changes are very sparse for the `Minimal` hierarchy, which must mean that the positive differences were wery low over all, to make the hierarchy achieve the lowest median error after reconciliation on daily granularity.

In [ ]:
for hier in (sth, mth, wth, wweth, wfth):
    plots.plot_error_diff(nrmse_em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth, granularity=MONTHS),
                                    MI(forecast_type=RECONCILED_FORECASTS, hierarchy=hier, granularity=MONTHS),
                                    outlier_threshold=0.995
    ).show()

The overall shapes look very similar for each hierarchy. We must remember that Minimal had no huge outliers to filter out.

In negative diffs, unlike at daily granularity, we can see how the maximum amount of negative difference was achieved consistently in each hierarchy, this is seen by tracing a -y=x axis, which the outline of the green dots follow.

Overall for both days and months, the minimal hierarchy's reconciliation introduced the least amount of outliers, with no extreme outlier at all. This makes sense. 

### Filtered stats
Now we check the statistics of the errors, but after filtering them for positive and negative values

In [ ]:
nrmse_diffs.get_error_stats(MI(granularity=DAYS), df_filter=lambda df: df[df > 0])

For positive only diffs at daily granularity, Minimal has the lowest mean and median and variance, as we already suspected from the plots and previous stats. This further reinforces the fact that `Minimal` is a consistent hierarchy. 

`Semantic` hierarchy has the worst median at daily granularity positive error diffs.

From the counts we can see that the lowest number of error increases at daily granularity happened with `Minimal` hierarchy.

In [ ]:
nrmse_diffs.get_error_stats(MI(granularity=DAYS), df_filter=lambda df: df[df < 0])

As for negative diffs on daily granularity, the lowest median is with `Semantic` hierarchy. The biggest value (so the worst median) is with the `Minimal` hierarchy.

`Minimal` has the biggest number of improved errors. We already saw from the plots that Minimal hierarchy had a consistent, but not very great performance at daily granularity.

We can see from the counts that `Semantic` had the lowest number of improvements in difference.

In [ ]:
nrmse_diffs.get_error_stats(MI(granularity=MONTHS), df_filter=lambda df: df[df > 0])

On monthly level, in terms of positive changes the medians are very similar across the different hierarchies. From the 90th percentiles `Semantic` seems the best. The worst q90 value is at the `Minimal` hierarchy.

It is interesting to see how consistent the number of positive diffs are, across all hierarchies, there are roughly 6400.

In [ ]:
nrmse_diffs.get_error_stats(MI(granularity=MONTHS), df_filter=lambda df: df[df < 0])

In terms of negative differences at monthly granularity (, so improvements), the medians look very similar still. From the 75th percentile we can see that `Minimal` performs the worst, and `Semantic` performs best.

### Difference sign distributions
In this section I will look at the heatmaps visualising the cases of the joint distribution of the sign of error differences at the monthly and daily granularity per each hierarchy.

In [ ]:
for h in (sth, mth, wth, wweth, wfth):
    plots.plot_error_signs(nrmse_diffs, MI(granularity=DAYS, hierarchy=f'{h} - Minimal'),
                                        MI(granularity=MONTHS, hierarchy=f'{h} - Minimal')
    ).show()

`Minimal` is the hierarchy which looks very different from the other hierarchies, which all look roughly the same. Lets compare `Semantic` and `Minimal` to demonstate:

| Daily Error Change | Monthly Error change | Semantic | Minimal |
| -------------------- | ------------------ | -------- | ------- |
| + | + | 2601 | 1275 |
| - | - | 1816 | 1204 |
| + | - | 4755 | 5310 |
| - | + | 3828 | 5206 |

It is clear that with the `Minimal` hierarchy the cases where the errors at both granularity increased were half as much as they were in `Semantic`. There were 50% more cases where both the errors decreased with `Semantic` hierarchy. There were roughly the same amount of time series where the signs of changes were (+,-) and (-,+) with the `Minimal` hierarchy. The cases where the daily error increased and monthly decreased were far more than cases where daily error decreased while monthly increased with the `Semantic` hierarchy. 

### Conclusions
Across all hierarchies, the median of error differences was slightly positive at daily granularity (meaning errors increased more often than not), and negative at monthly granularity (meaning errors improved more often than not). In both cases, Minimal was on the edge of these trends, meaning it had lowest absolute value medians. At daily granularity, the negative differences did not follow the x = -y line and were sparse and small in absolute value, unlike at monthly granularity where the negative differences consistently traced the x = -y axis. The count of positive error differences at monthly granularity was roughly the same across all hierarchies.

**Semantic**
- Pros
	- When looking at monthly granularity error differences, this hierarchy had the lowest median and the lowest quantiles above the 50th percentile, across all hierarchies.
	- When looking at negative (improved) daily error differences, this hierarchy had the lowest quantiles, meaning the deepest improvements at daily granularity. 
	- When looking at monthly positive error differences, this hierarchy had the lowest values from the 75th percentile upward.
	- When looking at monthly negative error differences, this hierarchy had the best (lowest) values from the 25th percentile downward.
- Cons
	- When looking at daily granularity error differences, this hierarchy had the highest median, across all hierarchies.
	- When looking at positive daily error differences, this hierarchy had the highest quantiles, meaning the largest error increases at daily granularity.
	- When looking at the count of positive daily error differences, this hierarchy had the highest count, meaning errors increased most frequently at daily granularity.
	- When looking at the count of negative daily error differences, this hierarchy had the lowest count, meaning the fewest improvements at daily granularity.

**Minimal**
- Pros
	- When looking at daily granularity error differences, this hierarchy had no outliers, as confirmed by the max and 90th percentile values.
	- When looking at daily granularity error differences, this hierarchy had the lowest quantiles and the lowest absolute values overall.
	- When looking at positive daily error differences, this hierarchy had the lowest quantiles and standard deviation, meaning the most contained error increases.
	- When looking at the count of positive daily error differences, this hierarchy had the lowest count, meaning errors increased least frequently at daily granularity.
	- When looking at the count of negative daily error differences, this hierarchy had the highest count, meaning the most improvements at daily granularity.
	- When looking at the error sign distributions, this hierarchy had the lowest number of cases where both daily and monthly errors increased (positive-positive).
- Cons
	- When looking at monthly granularity error differences, this hierarchy had the highest (worst) median, across all hierarchies.
	- When looking at negative daily error differences, this hierarchy had the highest (worst) quantiles, meaning the shallowest improvements at daily granularity.
	- When looking at monthly positive error differences, this hierarchy had the highest median.
	- When looking at monthly negative error differences, this hierarchy had the worst quantiles, meaning the shallowest monthly improvements.
	- When looking at the error sign distributions, this hierarchy had the lowest number of cases where both daily and monthly errors decreased (negative-negative).
	- When looking at the error sign distributions, this hierarchy had roughly equal counts of positive-negative and negative-positive sign changes, indicating many mixed-direction changes between daily and monthly granularity.


Highlights
- Error sign distributions between daily and monthly granularity
- Daily/Monthly reconciled vs. base error difference plots, especially the negative side
- Error difference statistics, especially plain, but also filtered positive/negative 